# RAG System Overview

This notebook builds a Retrieval-Augmented Generation (RAG) system that answers questions using information taken directly from research papers. The overall workflow includes:

1. Loading PDF files
2. Extracting text from the PDFs
3. Cleaning and chunking the text into segments
4. Generating vector embeddings for each chunk
5. Storing the embeddings in a ChromaDB vector database
6. Retrieving the most relevant chunks based on a user question
7. Sending the retrieved context to a language model to generate an answer
8. Returning an answer supported by source citations

The goal is to demonstrate a complete and functional RAG pipeline suitable for a capstone project.

In [1]:
import os

data_path = r"C:\Users\gayat\OneDrive\Desktop\capstone_rag\data"
print(os.listdir(data_path))


['attention_paper.pdf', 'gemini_paper.pdf', 'gpt4.pdf', 'instructgpt.pdf', 'mistral_paper.pdf']


## Imports and Setup

This section loads all required libraries and sets up project paths.


In [2]:
import os
import sys

print("OS and SYS imported successfully.")


OS and SYS imported successfully.


In [17]:
tests = {
    "pypdf": "pypdf",
    "pdfplumber": "pdfplumber",
    "numpy": "numpy",
    "pandas": "pandas"
}

for name, module in tests.items():
    try:
        __import__(module)
        print(f"{name}  imported successfully")
    except ImportError:
        print(f"{name}  not installed")


pypdf  imported successfully
pdfplumber  imported successfully
numpy  imported successfully
pandas  imported successfully


In [18]:
project_root = r"C:\Users\gayat\OneDrive\Desktop\capstone_rag"
print("Project root set to:", project_root)


Project root set to: C:\Users\gayat\OneDrive\Desktop\capstone_rag


In [19]:
import os

print(os.listdir(project_root))


['.ipynb_checkpoints', 'app', 'data', 'notebooks', 'README.txt']


In [20]:
try:
    import pypdf
    print("pypdf ✓")
except ImportError as e:
    print("pypdf ✗", e)

try:
    import pdfplumber
    print("pdfplumber ✓")
except ImportError as e:
    print("pdfplumber ✗", e)

try:
    import chromadb
    print("chromadb ✓")
except ImportError as e:
    print("chromadb ✗", e)

try:
    from sentence_transformers import SentenceTransformer
    print("sentence-transformers ✓")
except ImportError as e:
    print("sentence-transformers ✗", e)


pypdf ✓
pdfplumber ✓
chromadb ✓
sentence-transformers ✓


## Extracting Text
This section extracts raw text from each PDF file.

In [21]:
import os
import pdfplumber

data_path = r"C:\Users\gayat\OneDrive\Desktop\capstone_rag\data"
file_name = "attention_paper.pdf"  # exact name from your folder

pdf_path = os.path.join(data_path, file_name)
print("PDF path:", pdf_path)
print("Exists:", os.path.exists(pdf_path))


PDF path: C:\Users\gayat\OneDrive\Desktop\capstone_rag\data\attention_paper.pdf
Exists: True


In [22]:
text_pages = []

with pdfplumber.open(pdf_path) as pdf:
    # read first 3 pages safely (or fewer if document is short)
    num_pages = min(3, len(pdf.pages))
    print("Pages in PDF:", len(pdf.pages))
    print("Reading pages:", num_pages)
    
    for i in range(num_pages):
        page = pdf.pages[i]
        page_text = page.extract_text() or ""
        text_pages.append(page_text)

full_text_sample = "\n".join(text_pages)

print("Length of extracted text:", len(full_text_sample))
print("\n--- TEXT SAMPLE START ---\n")
print(full_text_sample[:1000])  # show first 1000 characters
print("\n--- TEXT SAMPLE END ---")


Pages in PDF: 15
Reading pages: 3
Length of extracted text: 7976

--- TEXT SAMPLE START ---

Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@google.com noam@google.com nikip@google.com usz@google.com
LlionJones∗ AidanN.Gomez∗ † ŁukaszKaiser∗
GoogleResearch UniversityofToronto GoogleBrain
llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com
IlliaPolosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
Thedominantsequencetransductionmodelsarebasedoncomplexrecurrentor
convolutionalneuralnetworksthatincludeanencoderandadecoder. Thebest
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
basedsolelyonattentionmechanisms,dispensingwithrecurrenceandco

In [23]:
from pypdf import PdfReader
import os

data_path = r"C:\Users\gayat\OneDrive\Desktop\capstone_rag\data"
file_name = "attention_paper.pdf"

pdf_path = os.path.join(data_path, file_name)

reader = PdfReader(pdf_path)
print("Total pages:", len(reader.pages))

page = reader.pages[0]  # first page
text_pypdf = page.extract_text()

print("\n--- PYPDF TEXT SAMPLE START ---\n")
print(text_pypdf[:1000])
print("\n--- PYPDF TEXT SAMPLE END ---")


Total pages: 15

--- PYPDF TEXT SAMPLE START ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispens

In [24]:
from pypdf import PdfReader
import os

def extract_text_pypdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = []
    
    for page in reader.pages:
        text = page.extract_text() or ""
        full_text.append(text)
    
    return "\n".join(full_text)


In [25]:
pdf_texts = {}

for filename in os.listdir(data_path):
    if filename.lower().endswith(".pdf"):
        file_path = os.path.join(data_path, filename)
        print(f"Reading: {filename}")
        
        text = extract_text_pypdf(file_path)
        pdf_texts[filename] = text
        
        print(f"  ✓ Extracted {len(text)} characters\n")

print("Done. Total PDFs processed:", len(pdf_texts))


Reading: attention_paper.pdf
  ✓ Extracted 39601 characters

Reading: gemini_paper.pdf
  ✓ Extracted 121381 characters

Reading: gpt4.pdf
  ✓ Extracted 38001 characters

Reading: instructgpt.pdf
  ✓ Extracted 76326 characters

Reading: mistral_paper.pdf
  ✓ Extracted 17637 characters

Done. Total PDFs processed: 5


In [26]:
import re

def clean_text(text):
    # Remove multiple spaces/newlines
    text = re.sub(r'\s+', ' ', text)
    
    # Remove weird unicode characters
    text = text.encode("ascii", "ignore").decode()
    
    # Strip leading/trailing spaces
    text = text.strip()
    
    return text


In [27]:
cleaned_texts = {}

for filename, text in pdf_texts.items():
    cleaned = clean_text(text)
    cleaned_texts[filename] = cleaned
    print(f"{filename}: cleaned length = {len(cleaned)}")


attention_paper.pdf: cleaned length = 39495
gemini_paper.pdf: cleaned length = 121102
gpt4.pdf: cleaned length = 37929
instructgpt.pdf: cleaned length = 75902
mistral_paper.pdf: cleaned length = 17590


In [28]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = start + (chunk_size - overlap)
    
    return chunks


In [29]:
all_chunks = []
doc_ids = []

for filename, text in cleaned_texts.items():
    chunks = chunk_text(text, chunk_size=1000, overlap=200)
    print(f"{filename}: {len(chunks)} chunks")
    
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        doc_ids.append(f"{filename}__chunk_{i}")

print("\nTotal chunks across all documents:", len(all_chunks))


attention_paper.pdf: 50 chunks
gemini_paper.pdf: 152 chunks
gpt4.pdf: 48 chunks
instructgpt.pdf: 95 chunks
mistral_paper.pdf: 22 chunks

Total chunks across all documents: 367


In [31]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name)

print("Model loaded:", embedding_model_name)

Model loaded: all-MiniLM-L6-v2


In [32]:
import numpy as np

embeddings = embedder.encode(all_chunks, show_progress_bar=True)
embeddings = np.array(embeddings)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embeddings shape: (367, 384)


In [53]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="C:/Users/gayat/OneDrive/Desktop/capstone_rag/chroma_db"
)

collection = chroma_client.get_collection(name="research_papers")
print("Collection loaded:", collection.name)


Collection loaded: research_papers


In [35]:
collection.add(
    ids=doc_ids,
    documents=all_chunks,
    embeddings=embeddings.tolist()
)

print("Inserted:", len(doc_ids), "chunks into ChromaDB")


Inserted: 367 chunks into ChromaDB


In [36]:
query = "What is the Transformer architecture?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

results


C:\Users\gayat\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:21<00:00, 3.82MiB/s]


{'ids': [['attention_paper.pdf__chunk_15',
   'attention_paper.pdf__chunk_30',
   'attention_paper.pdf__chunk_8']],
 'embeddings': None,
 'documents': [['layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways:  In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9].  The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of

In [39]:
def retrieve_context(question, k=3):
    """
    Returns:
      context (str): concatenated top-k chunks
      sources (list[str]): list of chunk IDs used
    """
    results = collection.query(
        query_texts=[question],
        n_results=k
    )
    
    docs = results["documents"][0]      # list of k chunks
    ids = results["ids"][0]             # list of k ids
    
    # Join chunks into one context string with separators
    context = "\n\n---\n\n".join(docs)
    
    return context, ids

In [40]:
question = "What is the Transformer architecture?"
context, sources = retrieve_context(question, k=3)

print("QUESTION:")
print(question)
print("\nSOURCES USED:")
for s in sources:
    print(" -", s)

print("\nCONTEXT PREVIEW (first 1000 characters):")
print(context[:1000])


QUESTION:
What is the Transformer architecture?

SOURCES USED:
 - attention_paper.pdf__chunk_15
 - attention_paper.pdf__chunk_30
 - attention_paper.pdf__chunk_8

CONTEXT PREVIEW (first 1000 characters):
layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways:  In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9].  The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come

In [43]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


## Full RAG Pipeline

This section combines retrieval and generation into a single function.


In [44]:
from groq import Groq
import os

os.environ["GROQ_API_KEY"] = "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"   # <-- replace ONLY this
groq_client = Groq()

print("Groq client initialized")


Groq client initialized


In [47]:
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say only: connected"}
    ],
    max_tokens=5
)

print(response.choices[0].message.content)


connected


In [48]:
def generate_answer(question, context):
    prompt = f"""
You are an AI assistant answering questions based only on the provided context.
Do NOT use any outside knowledge.

QUESTION:
{question}

CONTEXT:
{context}

Write a clear, correct, fluent answer.
    """

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0
    )

    return response.choices[0].message.content


## Testing the RAG System

This section tests the complete RAG pipeline with a sample question.


In [49]:
def rag_answer(question, k=3):
    # Step 1: retrieve context + sources
    context, sources = retrieve_context(question, k=k)

    # Step 2: generate clean answer
    answer = generate_answer(question, context)

    # Step 3: return everything
    return answer, sources


In [50]:
question = "What is the Transformer architecture?"
answer, sources = rag_answer(question, k=3)

print("ANSWER:\n")
print(answer)

print("\n\nSOURCES USED:")
for s in sources:
    print(" -", s)


ANSWER:

The Transformer architecture is a neural sequence transduction model that follows an encoder-decoder structure. It consists of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder maps an input sequence of symbol representations to a sequence of continuous representations, which are then used by the decoder to generate an output sequence of symbols one element at a time. The Transformer uses multi-head attention in three different ways: encoder-decoder attention, self-attention in the encoder, and self-attention in the decoder. This architecture is designed to mimic the typical encoder-decoder attention mechanisms in sequence-to-sequence models, but with the advantage of using self-attention to allow every position in the decoder to attend over all positions in the input sequence.


SOURCES USED:
 - attention_paper.pdf__chunk_15
 - attention_paper.pdf__chunk_30
 - attention_paper.pdf__chunk_8
